In [4]:
import pandas as pd
import requests
from datetime import datetime

/opt/anaconda3/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [5]:
ports = {
    "Los Angeles": {
        "state": "CA",
        "lat": 33.7405,
        "lon": -118.2775
    },
    "Long Beach": {
        "state": "CA",
        "lat": 33.7542,
        "lon": -118.2165
    },
    "New York/New Jersey": {
        "state": "NY/NJ",
        "lat": 40.6681,
        "lon": -74.0451
    },
    "Savannah": {
        "state": "GA",
        "lat": 32.0809,
        "lon": -81.0912
    },
    "Houston": {
        "state": "TX",
        "lat": 29.7604,
        "lon": -95.3698
    },
    "Seattle/Tacoma": {
        "state": "WA",
        "lat": 47.2529,
        "lon": -122.4443
    }
}

ports_df = pd.DataFrame.from_dict(ports, orient="index").reset_index()
ports_df.rename(columns={"index": "port_name"}, inplace=True)

ports_df

,port_name,state,lat,lon
0,Los Angeles,CA,33.7405,-118.2775
1,Long Beach,CA,33.7542,-118.2165
2,New York/New Jersey,NY/NJ,40.6681,-74.0451
3,Savannah,GA,32.0809,-81.0912
4,Houston,TX,29.7604,-95.3698
5,Seattle/Tacoma,WA,47.2529,-122.4443


In [6]:
project_note = """
This notebook collects monthly trade and weather-related data for a U.S. port-level
supply chain disruption risk prediction system.

The MVP focuses on major U.S. ports and creates a dataset that will later be used
to predict whether a port is at high disruption risk.
"""

print(project_note)


This notebook collects monthly trade and weather-related data for a U.S. port-level
supply chain disruption risk prediction system.

The MVP focuses on major U.S. ports and creates a dataset that will later be used
to predict whether a port is at high disruption risk.



In [7]:
IMPORTS_PORT_HS_URL = "https://api.census.gov/data/timeseries/intltrade/imports/porths"
EXPORTS_PORT_HS_URL = "https://api.census.gov/data/timeseries/intltrade/exports/porths"

In [8]:
params = {
    "get": "I_COMMODITY,I_COMMODITY_LDESC,GEN_VAL_MO,GEN_VAL_YR,CON_VAL_MO,CON_VAL_YR",
    "time": "2024-01"
}

response = requests.get(IMPORTS_PORT_HS_URL, params=params)

print(response.status_code)
print(response.text[:500])

200

<html>
    <head>
        <title>Missing Key</title>
    </head>
    <body>
        <p>
            A valid <em>key</em> must be included with each data API request.
            You my signup for one <a href="key_signup.html">here</a>.
        </p>
    </body>
</html>



In [9]:
CENSUS_API_KEY = "adea131aecdf5942110d809fd31832543a48a5af"

In [10]:
TEST_URL = "https://api.census.gov/data/timeseries/intltrade/imports/enduse"

params = {
    "get": "CTY_CODE,CTY_NAME,GEN_VAL_MO",
    "time": "2024-01",
    "key": "adea131aecdf5942110d809fd31832543a48a5af"
}

response = requests.get(TEST_URL, params=params)

print("Status Code:", response.status_code)

print(response.text[:500])

Status Code: 200
[["CTY_CODE","CTY_NAME","GEN_VAL_MO","time"],
["-","TOTAL FOR ALL COUNTRIES","253692671061","2024-01"],
["0003","EUROPEAN UNION","46462552767","2024-01"],
["0014","PACIFIC RIM COUNTRIES","79475175773","2024-01"],
["0017","CAFTA-DR","2476381523","2024-01"],
["0020","USMCA (NAFTA)","71358714212","2024-01"],
["0021","TWENTY LATIN AMERICAN REPUBLICS","50033637626","2024-01"],
["0022","OECD","157123490134","2024-01"],
["0023","NATO","78314756606","2024-01"],
["0024","LAFTA","47479015654","2024-01"],



In [11]:
data = response.json()

imports_df = pd.DataFrame(data[1:], columns=data[0])

imports_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time
0,-,TOTAL FOR ALL COUNTRIES,253692671061,2024-01
1,0003,EUROPEAN UNION,46462552767,2024-01
2,0014,PACIFIC RIM COUNTRIES,79475175773,2024-01
3,0017,CAFTA-DR,2476381523,2024-01
4,0020,USMCA (NAFTA),71358714212,2024-01


In [12]:
imports_df["GEN_VAL_MO"] = pd.to_numeric(imports_df["GEN_VAL_MO"], errors="coerce")
imports_df["time"] = pd.to_datetime(imports_df["time"])

imports_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time
0,-,TOTAL FOR ALL COUNTRIES,253692671061,2024-01-01
1,0003,EUROPEAN UNION,46462552767,2024-01-01
2,0014,PACIFIC RIM COUNTRIES,79475175773,2024-01-01
3,0017,CAFTA-DR,2476381523,2024-01-01
4,0020,USMCA (NAFTA),71358714212,2024-01-01


In [13]:
print(imports_df.shape)
print(imports_df.columns)

(248, 4)
Index(['CTY_CODE', 'CTY_NAME', 'GEN_VAL_MO', 'time'], dtype='str')


In [14]:
all_data = []

years = range(2020, 2026)
months = range(1, 13)

for year in years:
    for month in months:

        month_str = f"{year}-{month:02d}"

        params = {
            "get": "CTY_CODE,CTY_NAME,GEN_VAL_MO",
            "time": month_str,
            "key": "adea131aecdf5942110d809fd31832543a48a5af"
        }

        response = requests.get(
            "https://api.census.gov/data/timeseries/intltrade/imports/enduse",
            params=params
        )

        try:
            data = response.json()

            temp_df = pd.DataFrame(data[1:], columns=data[0])

            all_data.append(temp_df)

            print(f"Collected: {month_str}")

        except Exception as e:
            print(f"Failed: {month_str} -> {e}")

Collected: 2020-01
Collected: 2020-02
Collected: 2020-03
Collected: 2020-04
Collected: 2020-05
Collected: 2020-06
Collected: 2020-07
Collected: 2020-08
Collected: 2020-09
Collected: 2020-10
Collected: 2020-11
Collected: 2020-12
Collected: 2021-01
Collected: 2021-02
Collected: 2021-03
Collected: 2021-04
Collected: 2021-05
Collected: 2021-06
Collected: 2021-07
Collected: 2021-08
Collected: 2021-09
Collected: 2021-10
Collected: 2021-11
Collected: 2021-12
Collected: 2022-01
Collected: 2022-02
Collected: 2022-03
Collected: 2022-04
Collected: 2022-05
Collected: 2022-06
Collected: 2022-07
Collected: 2022-08
Collected: 2022-09
Collected: 2022-10
Collected: 2022-11
Collected: 2022-12
Collected: 2023-01
Collected: 2023-02
Collected: 2023-03
Collected: 2023-04
Collected: 2023-05
Collected: 2023-06
Collected: 2023-07
Collected: 2023-08
Collected: 2023-09
Collected: 2023-10
Collected: 2023-11
Collected: 2023-12
Collected: 2024-01
Collected: 2024-02
Collected: 2024-03
Collected: 2024-04
Collected: 2

In [15]:
trade_df = pd.concat(all_data, ignore_index=True)

trade_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time
0,-,TOTAL FOR ALL COUNTRIES,195796204072,2020-01
1,0003,EUROPEAN UNION,35071255143,2020-01
2,0014,PACIFIC RIM COUNTRIES,65080414292,2020-01
3,0017,CAFTA-DR,1786453267,2020-01
4,0020,USMCA (NAFTA),53489130119,2020-01


In [16]:
trade_df["GEN_VAL_MO"] = pd.to_numeric(
    trade_df["GEN_VAL_MO"],
    errors="coerce"
)

trade_df["time"] = pd.to_datetime(trade_df["time"])

trade_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time
0,-,TOTAL FOR ALL COUNTRIES,195796204072,2020-01-01
1,0003,EUROPEAN UNION,35071255143,2020-01-01
2,0014,PACIFIC RIM COUNTRIES,65080414292,2020-01-01
3,0017,CAFTA-DR,1786453267,2020-01-01
4,0020,USMCA (NAFTA),53489130119,2020-01-01


In [20]:

trade_df.to_csv(
    "/Users/navikamaglani/Documents/personal/supply_chain_disruption_ai/data/raw/us_trade_data.csv",
    index=False
)

print("Dataset saved!")

Dataset saved!
